In [ ]:
import tensorflow as tf
import os
import cv2
import matplotlib.pyplot as plt
import random
import zipfile
import json
import uuid
import pandas as pd
from PIL import Image
import numpy as np

In [ ]:
ls "/kaggle/input/facedetection"

In [ ]:
ls "/kaggle/input/facedetection/train"

## LOADING THE DATA

In [ ]:
train_images_dir = "/kaggle/input/facedetection/train/images"
train_labels_dir = "/kaggle/input/facedetection/train/labels"
validation_images_dir = "/kaggle/input/facedetection/validation/images"
validation_labels_dir = "/kaggle/input/facedetection/validation/labels"
test_images_dir = "/kaggle/input/facedetection/test/images"
test_labels_dir = "//kaggle/input/facedetection/test/labels"


In [ ]:
for dir_name,_,images in os.walk("/kaggle/input/facedetection/"):
    print(f"dirname {dir_name} folders: {len(_)} images:{len(images)} ")

In [ ]:
import os
import cv2
import json
import numpy as np
import tensorflow as tf

def load_image_and_label(image_path, labels_dir):
    """Loading image, resize image into a specific shape, loads corresponding label."""

    # Loading image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (224, 224))

    # image = image.astype(np.float32) / 255.0

    # Load label
    unique_identifier = os.path.splitext(os.path.basename(image_path))[0]
    label_file_path = os.path.join(labels_dir, f"{unique_identifier}.json")

    with open(label_file_path, "r") as f:
        label_data = json.load(f)

    bbox = label_data["bbox"]
    class_label = label_data["class"]

    return image, (np.array(bbox), np.array(class_label))

def create_dataset(images_dir, labels_dir, batch_size=8):
    """Create a TensorFlow dataset from image and label directories."""
    image_files = os.listdir(images_dir)
    image_paths = [os.path.join(images_dir, img) for img in image_files]

    images = []
    bboxes = []
    class_labels = []

    for image_path in image_paths:
        image, (bbox, class_label) = load_image_and_label(image_path, labels_dir)
        images.append(image)
        bboxes.append(bbox)
        class_labels.append(class_label)

    images = np.array(images)
    bboxes = np.array(bboxes)
    class_labels = np.array(class_labels)

    
    images = images.astype(np.float32)
    # images=images/255.
    bboxes = bboxes.astype(np.float32)
    class_labels = class_labels.astype(np.int32)

    
    labels = (class_labels, bboxes)

    # Creating TensorFlow dataset
    dataset = tf.data.Dataset.from_tensor_slices((images, labels))

    dataset = dataset.shuffle(buffer_size=1000).batch(batch_size)

    return dataset


In [ ]:

train_dataset = create_dataset(train_images_dir, train_labels_dir)
validation_dataset = create_dataset(validation_images_dir, validation_labels_dir)
test_dataset = create_dataset(test_images_dir, test_labels_dir)

#### How One batch looks like

In [ ]:


for images, labels in train_dataset.take(1):
    print(f"Images batch shape: {images.shape}")
    class_name,bbox = labels
    print(class_name.shape)
    print(bbox.shape)


## Visualising how my data and annotations look like

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
for images, labels in train_dataset.take(1):
  fig ,ax=plt.subplots(1,8,figsize=(30,20))
  for j in range(8):
    class_name,bbox = labels
    bbox =bbox.numpy()[j]
    x_min=bbox[0]*224
    y_min=bbox[1]*224
    x_max=bbox[2]*224
    y_max=bbox[3]*224
    rect = patches.Rectangle((x_min, y_min), x_max - x_min, y_max - y_min,
                              linewidth=2, edgecolor='red', facecolor='none')
    ax[j].add_patch(rect)
    ax[j].scatter(x_min ,y_min ,color="red")
    ax[j].scatter(x_max ,y_max ,color="blue")
    ax[j].text(x_min,y_min,"start-point",fontsize=12,ha="center" ,va="center")
    ax[j].text(x_max,y_max,"end-point",fontsize=12,ha="center" ,va="center")
    ax[j].imshow(images[j]/255.)
    ax[j].set_title(class_name[j].numpy())
    ax[j].axis("off")

  plt.show()



## Creating Model (Multi tasking CNN) MTCNN

In [ ]:
vgg= tf.keras.applications.VGG16(include_top=False)

In [ ]:
vgg.summary()

In [ ]:
import tensorflow as tf

def build_model():
    inputs = tf.keras.layers.Input(shape=(224, 224, 3))

    # Shared feature extractor (VGG16 base up to block4_pool)
    vgg = tf.keras.applications.VGG16(include_top=False, input_tensor=inputs)
    for layer in vgg.layers:
        layer.trainable = False
    for layer in vgg.layers[-10:]:
        layer.trainable = True
    
    shared_layers = tf.keras.Model(inputs=vgg.input, outputs=vgg.get_layer('block4_pool').output)(inputs)

    # Additional shared layers
    shared_layers = tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same')(shared_layers)
    shared_layers = tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same')(shared_layers)
    shared_layers = tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same')(shared_layers)
    shared_layers = tf.keras.layers.Conv2D(16, (3, 3), activation='relu', padding='same')(shared_layers)
    shared_layers = tf.keras.layers.BatchNormalization()(shared_layers)
    shared_layers = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(shared_layers)

    # Classification for finding if face exists or not
    class_conv = tf.keras.layers.Conv2D(16, (3, 3), activation='relu', padding='same')(shared_layers)
    class_branch = tf.keras.layers.GlobalMaxPooling2D()(class_conv)
    class_branch = tf.keras.layers.Dense(2048, activation='relu')(class_branch)
    class_output = tf.keras.layers.Dense(1, activation='sigmoid', name='class_output')(class_branch)

    # Regression seperated model for finding the 4 points to create box around the face
    reg_conv = tf.keras.layers.Conv2D(16, (3, 3), activation='relu', padding='same')(shared_layers)
    reg_branch = tf.keras.layers.GlobalMaxPooling2D()(reg_conv)
    reg_branch = tf.keras.layers.Dense(2048, activation='relu')(reg_branch)
    reg_output = tf.keras.layers.Dense(4, activation='sigmoid', name='reg_output')(reg_branch)

    # Combined Model
    facetracker = tf.keras.models.Model(inputs=inputs, outputs=[class_output, reg_output])
    return facetracker




In [ ]:
facetracker = build_model()
facetracker.summary()

## Defining loss functions and oprimizers

In [ ]:
batches_per_epoch = len(train_dataset)
print(batches_per_epoch)

#### Different loss functions

1. MSE (  (x_min-x_pred_min)^2 + (y_min - y_pred_min )^2 + (x_max - x_pred_max )^2 + (y_max - y_pred_max)^2  ) + (width_diff)^2 +(height_diff)^2
2. MAE
3. IOU (Intersection over Union)
4. GIOU (Generalised Intersection over Union)

**GIOU** works best in case of Object detection. why ?
**Reasearch paper**:[GIOU stanford](https://giou.stanford.edu/GIoU.pdf)

**Differences between all**:[Read here](https://learnopencv.com/iou-loss-functions-object-detection/#:~:text=CIoU%20loss%20function%20is%20better,faster%20convergence%20compared%20to%20GIoU.)

In [ ]:
initial_lr=0.0001
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=initial_lr,
    decay_steps=1000,
    decay_rate=0.9,
    staircase=True
)

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

#### Loss for classification task

In [ ]:

def compute_classification_loss(y_true, y_pred):
    """
    Computes binary cross-entropy classification loss.

    Args:
        y_true: Tensor of true labels, shape (batch_size,) or (batch_size, 1).
        y_pred: Tensor of predicted probabilities, shape (batch_size, 1).

    Returns:
        Loss value (scalar tensor).
    """
    # Ensure y_true is reshaped to match the shape of y_pred
    if y_true.shape[-1] != 1:
        y_true = tf.reshape(y_true, (-1, 1))

    # Binary cross-entropy loss function
    classification_loss_fn = tf.keras.losses.BinaryCrossentropy()

    # Compute loss
    loss = classification_loss_fn(y_true, y_pred)
    return loss


#### Loss for regression task (MSE)       (diffence in points)^2 + (difference in width)^2 + (difference in height)^2

In [ ]:
def localization_loss(y_true, yhat):
    delta_coord = tf.reduce_sum(tf.square(y_true[:,:2] - yhat[:,:2]))

    h_true = y_true[:,3] - y_true[:,1]
    w_true = y_true[:,2] - y_true[:,0]

    h_pred = yhat[:,3] - yhat[:,1]
    w_pred = yhat[:,2] - yhat[:,0]

    delta_size = tf.reduce_sum(tf.square(w_true - w_pred) + tf.square(h_true-h_pred))
    # tf.print(delta_coord + delta_size)
    return delta_coord + delta_size



regressloss = localization_loss

In [ ]:
# TensorBoard setup
log_dir = "logs/fit/"
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

# Learning rate scheduler callback
lr_scheduler_callback = tf.keras.callbacks.LearningRateScheduler(lr_schedule)
file_writer = tf.summary.create_file_writer(log_dir)

## Training 

In [ ]:
from tensorflow.keras.models import Model

class FaceTracker(Model):
    def __init__(self, eyetracker, **kwargs):
        super().__init__(**kwargs)
        self.model = eyetracker

    def compile(self, optimizer, class_loss, localization_loss, **kwargs):
        super().compile(**kwargs)
        self.class_loss_fn = compute_classification_loss
        self.localization_loss = localization_loss
        self.optimizer = optimizer
        self.train_metrics = {
            "total_loss": tf.keras.metrics.Mean(name="total_loss"),
            "class_loss": tf.keras.metrics.Mean(name="class_loss"),
            "regress_loss": tf.keras.metrics.Mean(name="regress_loss"),
        }
        self.val_metrics = {
            "total_loss": tf.keras.metrics.Mean(name="val_total_loss"),
            "class_loss": tf.keras.metrics.Mean(name="val_class_loss"),
            "regress_loss": tf.keras.metrics.Mean(name="val_regress_loss"),
        }

    def train_step(self, data):
        X, y = data
        # print(X)
        # print(y)
        with tf.GradientTape() as tape:
            classes, coords = self.model(X, training=True)
            # print(classes.shape)
            # print(coords.shape)

            batch_class_loss = self.class_loss_fn(y[0], classes)
            batch_localization_loss = self.localization_loss(tf.cast(y[1], tf.float32), coords)
            total_loss = batch_localization_loss + 0.5 * batch_class_loss

        gradients = tape.gradient(total_loss, self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.model.trainable_variables))

        # Update metrics
        self.train_metrics["total_loss"].update_state(total_loss)
        self.train_metrics["class_loss"].update_state(batch_class_loss)
        self.train_metrics["regress_loss"].update_state(batch_localization_loss)

        return {name: metric.result() for name, metric in self.train_metrics.items()}

    def test_step(self, data):
        X, y = data
        classes, coords = self.model(X, training=False)
        batch_class_loss = self.class_loss_fn(y[0], classes)
        batch_localization_loss = self.localization_loss(tf.cast(y[1], tf.float32), coords)
        total_loss = batch_localization_loss + 0.5 * batch_class_loss

        # Update metrics
        self.val_metrics["total_loss"].update_state(total_loss)
        self.val_metrics["class_loss"].update_state(batch_class_loss)
        self.val_metrics["regress_loss"].update_state(batch_localization_loss)

        return {name: metric.result() for name, metric in self.val_metrics.items()}

    def reset_metrics(self):
        for metric in self.train_metrics.values():
            metric.reset_state()  # Corrected method
        for metric in self.val_metrics.values():
            metric.reset_state()  # Corrected method


    def call(self, X, **kwargs):
        return self.model(X, **kwargs)


In [ ]:

#creating a model

model = FaceTracker(facetracker)

#2. compiling a model
model.compile(
    tf.keras.optimizers.Adam(learning_rate=0.0001), compute_classification_loss,
       regressloss
)



logdir='logs'
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=logdir)

# 3. fitting a model
hist = model.fit(train_dataset, epochs=15, validation_data=validation_dataset, callbacks=[tensorboard_callback])

In [ ]:
def plot_loss_curves(history):
    fig ,ax = plt.subplots(1,3,figsize=(15,5))

    class_loss= history.history["class_loss"]
    val_class_loss = history.history["val_class_loss"]

    regress_loss = history.history["regress_loss"]
    val_regress_loss = history.history["val_regress_loss"]

    total_loss = history.history["total_loss"]
    val_total_loss = history.history["val_total_loss"]

    ax[0].plot(class_loss, color="red", label="class_loss")
    ax[0].plot(val_class_loss, color="blue", label="val_class_loss")
    ax[0].set_xlabel("Epochs")
    ax[0].set_ylabel("classification_loss")
    ax[0].legend()

    ax[1].plot(regress_loss, color="red", label="regress_loss")
    ax[1].plot(val_regress_loss, color="blue", label="val_regress_loss")
    ax[1].set_xlabel("Epochs")
    ax[1].set_ylabel("regression_loss")
    ax[1].legend()

    ax[2].plot(total_loss, color="red", label="total_loss")
    ax[2].plot(val_total_loss, color="blue", label="val_total_loss")
    ax[2].set_xlabel("Epochs")
    ax[2].set_ylabel("total_loss")
    ax[2].legend()



    plt.show()



## Plotting different curves (regression , classification and totol loss)

In [ ]:
plot_loss_curves(hist)

## Testing model 

In [ ]:
classes , bboxes =model.predict(test_dataset)
print(classes.shape)
print(bboxes.shape)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

test_data = test_dataset.as_numpy_iterator()
test_sample = test_data.next()

yhat = model.predict(test_sample[0])

fig, ax = plt.subplots(ncols=4, figsize=(20, 20))

for idx in range(4):
    sample_image = test_sample[0][idx]
    sample_coords = yhat[1][idx]
    sample_class = yhat[0][idx]


    print(f"Sample {idx} coordinates: {sample_coords}")

    x_min, y_min = np.multiply(sample_coords[:2], [224, 224]).astype(int)
    x_max, y_max = np.multiply(sample_coords[2:], [224, 224]).astype(int)

    # Drawing the bounding box
    sample_image_with_bbox = sample_image.copy()
    if sample_class[0]>0.5:
      sample_image_with_bbox = cv2.rectangle(
          sample_image_with_bbox,
          (x_min, y_min),
          (x_max, y_max),
          (255, 0, 0),  # Red color (BGR)
          2  # Thickness of the rectangle
      )


    # Creatung figurwe
    ax[idx].imshow(sample_image_with_bbox/255.)


    ax[idx].set_title(f"Image {idx+1} {sample_class} with Bounding Box")


    ax[idx].axis('off')
    ax[idx].set_title(sample_class[0])

plt.tight_layout()
plt.show()


In [ ]:
facetracker.save("baseline5.keras")